# Accelerating Large Scale Text Analysis Using Parallel Computing

## Sequential Implementation

Course: ARTI 503 – Parallel Architecture and Programming

This notebook implements the sequential version of the text analysis algorithm using the Amazon Fine Food Reviews dataset.
[dataset link](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews?resource=download)

In [2]:
import pandas as pd
import re
import time
from collections import Counter

In [3]:
%pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

print("Path to dataset files:", path)

100%|████████████████████████████████████████| 242M/242M [00:21<00:00, 12.0MB/s]

Extracting files...


Path to dataset files: /Users/frfooram/.cache/kagglehub/datasets/snap/amazon-fine-food-reviews/versions/2


In [5]:
# Load the Amazon Fine Food Reviews dataset
import pandas as pd
df = pd.read_csv(path+"/Reviews.csv")
reviews = df["Text"].dropna().astype(str).tolist()
print("Total reviews:", len(reviews))
print("\nFirst review:")
print(reviews[0][:300])


Total reviews: 568454

First review:
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.


# Sequential Algorithm

In [6]:
# Precompile the tokenization pattern
TOKEN_PATTERN = re.compile(r"\b\w+\b")

# Keywords selected for analysis
KEYWORDS = {"good", "bad", "excellent", "poor"}


def sequential_text_analysis(reviews):
    """
    Analyze reviews sequentially.

    The function performs:
    - Tokenization
    - Word frequency counting
    - Keyword counting
    - Basic text statistics
    """

    word_counts = Counter()
    keyword_counts = Counter()
    total_words = 0

    # Process reviews one at a time
    for review in reviews:

        # Convert to lowercase and tokenize
        words = TOKEN_PATTERN.findall(review.lower())

        # Calculate total number of words
        total_words += len(words)

        # Update global word frequencies
        word_counts.update(words)

        # Count selected keywords
        keyword_counts.update(
            word for word in words if word in KEYWORDS
        )

    total_reviews = len(reviews)
    unique_words = len(word_counts)

    average_words = (
        total_words / total_reviews
        if total_reviews > 0
        else 0
    )

    return {
        "total_reviews": total_reviews,
        "total_words": total_words,
        "unique_words": unique_words,
        "average_words": average_words,
        "word_frequencies": word_counts,
        "keyword_counts": keyword_counts
    }


# Test the Sequential Algorithm

In [7]:
# Small manually verified sample
test_reviews = [
    "Good food",
    "Good quality",
    "Bad food"
]

test_result = sequential_text_analysis(test_reviews)

assert test_result["total_reviews"] == 3
assert test_result["total_words"] == 6
assert test_result["unique_words"] == 4
assert test_result["average_words"] == 2.0
assert test_result["word_frequencies"]["good"] == 2
assert test_result["word_frequencies"]["food"] == 2
assert test_result["keyword_counts"]["good"] == 2
assert test_result["keyword_counts"]["bad"] == 1

print("Correctness test passed successfully.")

Correctness test passed successfully.


# Full Dataset Execution

In [8]:
import time

# Use the entire dataset
all_reviews = reviews

start_time = time.perf_counter()

results_all = sequential_text_analysis(all_reviews)

end_time = time.perf_counter()
execution_time_all = end_time - start_time

print("Total Reviews:", results_all["total_reviews"])
print("Total Words:", results_all["total_words"])
print("Unique Words:", results_all["unique_words"])
print("Average Words:", round(results_all["average_words"], 2))
print("Top 10 Words:", results_all["word_frequencies"].most_common(10))
print("Keyword Counts:", results_all["keyword_counts"])
print("Execution Time:", round(execution_time_all, 4), "seconds")

Total Reviews: 568454
Total Words: 47314867
Unique Words: 120297
Average Words: 83.23
Top 10 Words: [('the', 1872601), ('i', 1691844), ('and', 1289768), ('a', 1235964), ('it', 1073835), ('to', 1015494), ('of', 802081), ('is', 734366), ('this', 674782), ('br', 647089)]
Keyword Counts: Counter({'good': 200638, 'bad': 32186, 'excellent': 23101, 'poor': 3672})
Execution Time: 8.9897 seconds


## Operation Timing Analysis

In [9]:

import time
import pandas as pd
from collections import Counter

def analyze_operation_times(sample):
    word_counts = Counter()
    keyword_counts = Counter()

    tokenization_time = 0.0
    frequency_time = 0.0
    keyword_time = 0.0

    for review in sample:

        # 1. Tokenization
        start = time.perf_counter()
        words = TOKEN_PATTERN.findall(review.lower())
        tokenization_time += time.perf_counter() - start

        # 2. Word Frequency Counting
        start = time.perf_counter()
        word_counts.update(words)
        frequency_time += time.perf_counter() - start

        # 3. Keyword Counting
        start = time.perf_counter()
        keyword_counts.update(
            word for word in words if word in KEYWORDS
        )
        keyword_time += time.perf_counter() - start

    return {
        "Tokenization (s)": tokenization_time,
        "Word Frequency (s)": frequency_time,
        "Keyword Counting (s)": keyword_time
    }


# Operation Timing Analysis
input_sizes = [10000, 50000, 100000, 300000, len(reviews)]

operation_results = []

for size in input_sizes:
    sample = reviews[:size]

    print(f"Processing {len(sample):,} reviews...", flush=True)

    result = analyze_operation_times(sample)

    operation_results.append({
        "Input Size": len(sample),
        **result
    })

    print(f"Finished {len(sample):,} reviews!", flush=True)

operation_df = pd.DataFrame(operation_results)

print("\n===== Operation Timing Analysis =====")
display(operation_df)


# Bottleneck Analysis
operations = [
    "Tokenization (s)",
    "Word Frequency (s)",
    "Keyword Counting (s)"
]

bottleneck_results = []

for _, row in operation_df.iterrows():
    total_operation_time = sum(row[op] for op in operations)

    slowest_operation = max(operations, key=lambda op: row[op])
    slowest_time = row[slowest_operation]

    percentage = (
        slowest_time / total_operation_time * 100
        if total_operation_time > 0 else 0
    )

    bottleneck_results.append({
        "Input Size": int(row["Input Size"]),
        "Slowest Operation": slowest_operation.replace(" (s)", ""),
        "Time (s)": round(slowest_time, 4)
    })

bottleneck_df = pd.DataFrame(bottleneck_results)

print("\n===== Bottleneck Analysis =====")
display(bottleneck_df)


Processing 10,000 reviews...
Finished 10,000 reviews!
Processing 50,000 reviews...
Finished 50,000 reviews!
Processing 100,000 reviews...
Finished 100,000 reviews!
Processing 300,000 reviews...
Finished 300,000 reviews!
Processing 568,454 reviews...
Finished 568,454 reviews!

===== Operation Timing Analysis =====


,Input Size,Tokenization (s),Word Frequency (s),Keyword Counting (s)
0,10000,0.087672,0.041445,0.020207
1,50000,0.459396,0.218582,0.103394
2,100000,0.945223,0.442519,0.211379
3,300000,2.838403,1.345157,0.639304
4,568454,5.216666,2.445229,1.184536



===== Bottleneck Analysis =====


,Input Size,Slowest Operation,Time (s)
0,10000,Tokenization,0.0877
1,50000,Tokenization,0.4594
2,100000,Tokenization,0.9452
3,300000,Tokenization,2.8384
4,568454,Tokenization,5.2167


,Input Size,Slowest Operation,Time (s)
0,10000,Tokenization,0.3270
1,50000,Tokenization,1.4638
2,100000,Tokenization,3.0287
3,300000,Tokenization,8.7458
4,568454,Tokenization,18.3213



## Input Size Verification

The sequential algorithm is tested on different input sizes to confirm that it processes each dataset size correctly without using parallel computing. The tested input sizes are 10,000, 50,000, 100,000, 300,000, and 568,454 reviews.



In [10]:

import time
import statistics
import pandas as pd

input_sizes = [10000, 50000, 100000, 300000, len(reviews)]
repetitions = 3

benchmark_results = []

for size in input_sizes:
    sample = reviews[:size]
    execution_times = []

    print(f"Processing {len(sample):,} reviews...", flush=True)

    for run in range(repetitions):
        start = time.perf_counter()
        result = sequential_text_analysis(sample)
        elapsed = time.perf_counter() - start

        execution_times.append(elapsed)

    benchmark_results.append({
        "Input Size": len(sample),
        "Run 1": execution_times[0],
        "Run 2": execution_times[1],
        "Run 3": execution_times[2],
        "Average Time (s)": statistics.mean(execution_times)
    })

    print(f"Finished {len(sample):,} reviews!", flush=True)

benchmark_df = pd.DataFrame(benchmark_results)

print("\n===== Sequential Benchmark Results =====")
display(benchmark_df)

# Save only Input Size and Average Time
summary_df = benchmark_df[["Input Size", "Average Time (s)"]].copy()

print("\n===== Average Execution Times =====")
display(summary_df)


Processing 10,000 reviews...
Finished 10,000 reviews!
Processing 50,000 reviews...
Finished 50,000 reviews!
Processing 100,000 reviews...
Finished 100,000 reviews!
Processing 300,000 reviews...
Finished 300,000 reviews!
Processing 568,454 reviews...
Finished 568,454 reviews!

===== Sequential Benchmark Results =====


,Input Size,Run 1,Run 2,Run 3,Average Time (s)
0,10000,0.153950,0.151021,0.149502,0.151491
1,50000,0.788336,0.786410,0.791465,0.788737
2,100000,1.610861,1.622910,1.603976,1.612582
3,300000,4.747014,4.818367,4.848230,4.804537
4,568454,8.953288,9.025235,8.989427,8.989317



===== Average Execution Times =====


,Input Size,Average Time (s)
0,10000,0.151491
1,50000,0.788737
2,100000,1.612582
3,300000,4.804537
4,568454,8.989317
